In [2]:
# check langchain version it should be 0.3 if not uncomment below pip command to install the correct version
import langchain, langchain_community
print(langchain.__version__)
print(langchain_community.__version__)

0.3.27
0.3.24


In [3]:
#!pip uninstall langchain -y && pip uninstall langchain-core -y && pip uninstall langchain-community -y
#!pip install langchain==0.3.27 langchain-openai==0.3.33 langchain-community==0.3.24


# In this example: An Insurance Claims Assistant we using mock api tool for check claim status, Fraud Detection Tool, Escalation tool



In [14]:
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_core.tools import tool

# Load environment variables
load_dotenv()

# Create model
llm = ChatOpenAI(model="gpt-4o-mini")

In [5]:
from langchain.tools import tool

# Mock API Tool: Check Claim Status
@tool
def check_claim_status(claim_id: str) -> str:
    """Return status of a claim with given claim_id."""
    fake_db = {
        "C101": "Approved - Payment in process",
        "C202": "Under review - Waiting documents",
        "C303": "Flagged for manual investigation"
    }
    return fake_db.get(claim_id, "Claim ID not found")

# Fraud Detection Tool
@tool
def fraud_risk_check(description: str) -> str:
    """Returns risk evaluation for fraud based on description."""
    if "lost phone but still using it" in description.lower():
        return "High Risk"
    return "Low Risk"

# Escalation Tool
@tool
def escalate_to_human(issue: str) -> str:
    """Escalates conversation to a human agent."""
    return f"Escalation created. A human agent will review: {issue}"


In [19]:
from langchain_openai import ChatOpenAI
from langchain.agents import create_tool_calling_agent
from langchain.agents import AgentExecutor 
from langchain.prompts import ChatPromptTemplate


tools = [check_claim_status, fraud_risk_check, escalate_to_human]

system_prompt = ChatPromptTemplate.from_template("""
You are an Insurance Claims Assistant.

Your job:
1) Understand user intent.
2) If they ask about a claim status → call `check_claim_status`
3) If user mentions incident details → call `fraud_risk_check`
4) If unsure, conflicting, or user upset → call `escalate_to_human`

Always respond in a friendly, formal tone.
user request: {input},
"placeholder", "{agent_scratchpad}"
"""
)



In [22]:
from openai import max_retries


agent = create_tool_calling_agent(
    llm=llm,
    tools=tools,
    prompt=system_prompt
)
executor = AgentExecutor.from_agent_and_tools(agent=agent, tools=tools, verbose=True)


In [23]:
# Test the agent with a sample query
query = "What's the current status of claim C202?"
result = executor.invoke({"input": query})
print("Agent Output:", result)



> Entering new AgentExecutor chain...

Invoking: `check_claim_status` with `{'claim_id': 'C202'}`


Under review - Waiting documentsThe current status of your claim C202 is: **Under review - Waiting for documents**. If you have any further questions or need assistance, please feel free to ask!

> Finished chain.
Agent Output: {'input': "What's the current status of claim C202?", 'output': 'The current status of your claim C202 is: **Under review - Waiting for documents**. If you have any further questions or need assistance, please feel free to ask!'}


In [25]:
#test fraud_risk_check tools: "I lost my phone but I'm still using it. Is that normal?"
#test escalate_to_human: "Your bot is useless. I want a human NOW!"
